# Sentiment Analysis using Bidirectional GRU with GloVe

## CSE 4221 — Natural Language Processing Assignment

**Model:** Bidirectional GRU  
**Embeddings:** GloVe (300-dimensional, pre-trained)  
**Dataset:** IMDB Movie Review Dataset  
**Task:** Binary Sentiment Classification (Positive / Negative)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, GRU, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported successfully.')

---
## 2. Load the IMDB Dataset

In [ ]:
df = pd.read_csv('IMDB-Dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nSentiment Distribution:\n{df["sentiment"].value_counts()}')

# Remove duplicates
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed {duplicates} duplicates. New shape: {df.shape}')

df.head()

---
## 3. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean and preprocess a single review."""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

print('Preprocessing reviews...')
df['cleaned_review'] = df['review'].apply(preprocess_text)
print('Preprocessing complete.')

# Encode labels
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(f'\nSample cleaned review:\n{df["cleaned_review"].iloc[0][:150]}')

---
## 4. Train-Test Split (80:20)

In [ ]:
X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)}')
print(f'Test set:     {len(X_test)}')

---
## 5. Tokenization and Sequence Padding

In [ ]:
# Parameters
VOCAB_SIZE = 20000
MAX_LEN = 200
EMBED_DIM = 300

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert to sequences and pad
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

word_index = tokenizer.word_index
print(f'Vocabulary size: {len(word_index)}')
print(f'Training shape: {X_train_pad.shape}')
print(f'Test shape:     {X_test_pad.shape}')

---
## 6. Load Pre-trained GloVe Embeddings

We use **GloVe** (Global Vectors for Word Representation) pre-trained on large corpora. Download `glove.6B.300d.txt` from [https://nlp.stanford.edu/projects/glove/](https://nlp.stanford.edu/projects/glove/) and place it in the working directory.

Unlike Word2Vec trained on our small corpus, GloVe provides pre-trained embeddings from a much larger corpus (6B tokens from Wikipedia + Gigaword).

In [ ]:
# If GloVe file is not present, download it automatically
import urllib.request
import zipfile

GLOVE_FILE = 'glove.6B.300d.txt'
GLOVE_ZIP = 'glove.6B.zip'

if not os.path.exists(GLOVE_FILE):
    print('GloVe file not found. Downloading glove.6B.zip (~862 MB)...')
    print('This may take several minutes depending on your connection.')
    url = 'https://nlp.stanford.edu/data/glove.6B.zip'
    urllib.request.urlretrieve(url, GLOVE_ZIP)
    print('Download complete. Extracting...')
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as zip_ref:
        zip_ref.extract(GLOVE_FILE)
    print('Extraction complete.')
else:
    print(f'GloVe file found: {GLOVE_FILE}')

In [ ]:
# Load GloVe embeddings into a dictionary
print('Loading GloVe embeddings...')
glove_embeddings = {}
with open(GLOVE_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

print(f'Total GloVe word vectors loaded: {len(glove_embeddings)}')
print(f'Embedding dimension: {len(list(glove_embeddings.values())[0])}')

In [ ]:
# Build embedding matrix from GloVe
vocab_size = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

found = 0
not_found = 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    vector = glove_embeddings.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
        found += 1
    else:
        not_found += 1

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Words found in GloVe:     {found}')
print(f'Words NOT found:          {not_found}')
print(f'Coverage: {found/(found+not_found)*100:.1f}%')

---
## 7. Build Bidirectional GRU Model

GRU (Gated Recurrent Unit) is a streamlined variant of LSTM with fewer parameters. It uses reset and update gates instead of LSTM's three gates, making it faster to train while maintaining comparable performance.

In [ ]:
model = Sequential([
    # Pre-trained GloVe embeddings (non-trainable)
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    SpatialDropout1D(0.3),
    
    # Bidirectional GRU layers
    Bidirectional(GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Bidirectional(GRU(64, dropout=0.2, recurrent_dropout=0.2)),
    
    # Dense classification layers
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## 8. Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

y_train_arr = y_train.values
y_test_arr = y_test.values

history = model.fit(
    X_train_pad, y_train_arr,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Prediction and Evaluation

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy  = accuracy_score(y_test_arr, y_pred)
precision = precision_score(y_test_arr, y_pred)
recall    = recall_score(y_test_arr, y_pred)
f1        = f1_score(y_test_arr, y_pred)

print('=' * 55)
print('  Bi-GRU + GloVe — RESULTS')
print('=' * 55)
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1-Score:  {f1:.4f}')
print('=' * 55)

In [ ]:
print('\nClassification Report:')
print(classification_report(y_test_arr, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_arr, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap='Purples', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — Bi-GRU + GloVe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Analysis and Discussion

### Observations
- **GloVe embeddings** (pre-trained on 6B tokens) provide high-quality word representations with broad vocabulary coverage.
- **Bidirectional GRU** captures sequential context from both directions with fewer parameters than LSTM.
- The stacked architecture allows hierarchical feature learning from word-level to phrase-level patterns.

### GloVe vs Word2Vec
- GloVe is trained on global co-occurrence statistics, capturing broader semantic relationships.
- Pre-trained on a much larger corpus (6B tokens) vs Word2Vec trained only on IMDB data.
- Higher vocabulary coverage translates to fewer zero vectors in the embedding matrix.

### GRU vs LSTM
- GRU has fewer parameters (2 gates vs 3) making it faster to train.
- Often achieves comparable performance to LSTM on many NLP tasks.
- Simpler architecture may generalize better on smaller datasets.

### Conclusion
The Bi-GRU + GloVe combination is expected to perform competitively with or better than Bi-LSTM + Word2Vec, owing to the superior pre-trained embeddings and the efficient GRU architecture.